### This Notebooks Contains the Code For Learning Stock Embeddings
This approach was described in:
- The original preprint: [Stock Embeddings: Learning Distributed Representations for Financial Assets](https://arxiv.org/pdf/2202.08968.pdf)
- The more updated conference publication which can be found [here](https://www.mdpi.com/2673-4591/39/1/30/pdf?version=1695109017)

For any questions, reach out to me at dolphrian@gmail.com

In [1]:
from stock_embeddings import ReturnsData
from stock_embeddings import get_pointwise_tgt_context_sets

# Constants and configurations
PERIODS = [1, 5, 10]
TRAIN_PCT = 1
CONTEXT_SIZE = 32

tgt_context_sets = []
for period in PERIODS:
    print(f"Running for period {period}")
    data = ReturnsData()
    data.change_returns_period(period)
    data.train_test_split(TRAIN_PCT)

    tgt_context_sets += get_pointwise_tgt_context_sets(
        data.train_returns_df.select(data.tickers),
        context_size=CONTEXT_SIZE,
        verbose=True,
        iqr_noise_reduction=True,
    )
    print("="*20)
print(f"Total number of target context sets: {len(tgt_context_sets)}")

Running for period 1


No change made because period entered is 1


  0%|          | 0/611 [00:00<?, ?it/s]

  2%|▏         | 13/611 [00:00<00:04, 125.18it/s]

  5%|▍         | 28/611 [00:00<00:04, 125.14it/s]

...

Running for period 10


  0%|          | 0/611 [00:00<?, ?it/s]

 88%|████████▊ | 540/611 [00:00<00:00, 5397.19it/s]

100%|██████████| 611/611 [00:00<00:00, 4679.70it/s]

Total number of target context sets: 1860195


In [2]:
import polars as pl
import numpy as np

ticker = "JPM"
i = data.ticker2idx[ticker]
temp = np.array([xi[1] for xi in tgt_context_sets if xi[0] == i]).flatten()
top_values = pl.Series("value", temp).value_counts(sort=True)["value"].to_list()
print(
    f"The most commonly co-occurring stocks with {ticker} are: {[data.idx2ticker[xi] for xi in top_values][:5]}"
)

The most commonly co-occurring stocks with JPM are: ['C', 'BAC', 'STI', 'WFC', 'PNC']


In [3]:
from stock_embeddings import ClassificationEmbeddings

EMBEDDING_DIM = 20
model = ClassificationEmbeddings(
    n_time_series=len(data.tickers), embedding_dim=EMBEDDING_DIM
)

In [4]:
from stock_embeddings import train_embeddings_from_tgt_context_sets

EPOCHS = 10

model, losses = train_embeddings_from_tgt_context_sets(
    n_time_series=len(data.tickers),
    tgt_context_sets=tgt_context_sets,
    model=model,
    epochs=EPOCHS,
    batch_size=64,
    early_stopping=False,
    device="cpu",
    # embedding_dim=EMBEDDING_DIM,
    verbose=True,
)

Training embeddings...


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:17<02:35, 17.29s/it]

Epoch 0: Loss = 0.10368980767777962


 20%|██        | 2/10 [00:34<02:17, 17.23s/it]

...

 90%|█████████ | 9/10 [02:34<00:17, 17.17s/it]

Epoch 8: Loss = 0.09736572647415667


100%|██████████| 10/10 [02:51<00:00, 17.18s/it]

100%|██████████| 10/10 [02:51<00:00, 17.19s/it]

Epoch 9: Loss = 0.09729221038419081


In [5]:
from stock_embeddings import get_sector_score

get_sector_score(model.embeddings.weight.detach().numpy(), sectors=data.sectors, top_k_accuracy=True)

/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=Tru

/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Precision Score: 0.67
Recall Score: 0.65
F1 Score: 0.65
Accuracy Score: 0.65
Accuracy Score Top-3: 0.86


/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [6]:
from stock_embeddings.plotting import pca_plot_from_embeddings

pca_plot_from_embeddings(
    embedding_matrix=model.embeddings.weight.detach().numpy(),
    sectors=data.sectors,
    tickers=data.tickers,
    industries=data.industries,
    names=data.names,
    dimensions=2,
    reduced=True,
    method="PCA",
    return_df=False,
    rand_state=None,
)

In [7]:
SAVE_MODEL = False
SAVE_PATH_TEMPLATE = "abs_diff_E{epochs}_C{context_size}_D{embedding_dim}_P{periods}_train{train_pct}.pt"
save_path = SAVE_PATH_TEMPLATE.format(
    epochs=EPOCHS,
    context_size=CONTEXT_SIZE,
    embedding_dim=EMBEDDING_DIM,
    periods="-".join(map(str, PERIODS)),
    train_pct=TRAIN_PCT,
)

# Example usage of saving and loading model
# model.save_model("model_test.pt")
# model = ClassificationEmbeddings.load_model(path="model_test.pt")